## inference for proportions

In [77]:
# _rd_00.py
# 비율 추론. 해보자. phat, se, 신뢰구간, 검정통계량, pvalue.

import os
import numpy as np                          # numpy 라이브러리 전체.
import pandas as pd                         # pandas 라이브러리 전체, 시계열(기본) 여기 있네.
import scipy as sci
from scipy import stats, optimize, linalg   # scipy 라이브러리 하부 모듈. 필요한 통계 모듈만.
import statsmodels.api as sm                # 방대한 모델이라 개발자들이 많이 쓰는 모델 묶음.
from statsmodels.tsa import stattools       # time series analysis
from statsmodels.tsa.arima.model import ARIMA # 이거 가능하다 이거지. 이거 대문자네. 꼭. 소문자 못읽어.
import matplotlib.pyplot as plt
from statsmodels.stats.proportion import proportion_confint, proportions_ztest


### 원격 데이터 파일(엑셀, csv 파일, 인터넷) 읽기

In [78]:
# 1. 데이터 준비. 읽기. 생성.
# 데이터 파일 읽기. github.com
dat_url = 'https://github.com/bahn28/gamja/blob/main/cs_nns_gndr_hgt.csv?raw=true'
df_dat = pd.read_csv(dat_url)
df_dat.head()

,i,gender,ht
0,1,1,159.9
1,2,2,157.5
2,3,2,158.0
3,4,2,154.2
4,5,1,163.3


### 변수 생성, 리네임

In [79]:
df_dat['gn'] = df_dat['gender'].to_numpy()
gn = df_dat['gn']
#gn = df_dat['gender'].to_numpy()   # 이렇게 하면 배열로 전환. dataframe -> NumPy 배열
df_dat['hgt'] = df_dat['ht'].to_numpy()
hgt = df_dat['hgt']
#hgt = df_dat['ht'].to_numpy()
df_dat

,i,gender,ht,gn,hgt
0,1,1,159.9,1,159.9
1,2,2,157.5,2,157.5
2,3,2,158.0,2,158.0
3,4,2,154.2,2,154.2
4,5,1,163.3,1,163.3
...,...,...,...,...,...
20123,20124,2,157.3,2,157.3
20124,20125,1,175.9,1,175.9
20125,20126,1,175.6,1,175.6
20126,20127,2,158.0,2,158.0


### 자료셋, 그룹 구분
 전체, (Tall 그룹, Short 그룹), (Male, Female).

키 구분은 전체 평균을 기준으로 함.

In [80]:
# 데이터 정비, 그룹별 분리 등.
# Total, Tall, Short. Three 3 data sets.

# 2. groups: by gndr (gender, 1 - 0), by hgt_grp (height, Tall - Short)
gndr = (gn == 1).astype(int)    # gn = 1, 2; gndr = 1, 0. 1/0으로 전환.

hgt_ref = np.mean(hgt)          # 그룹 구분 참고값. 평균 이상, 이하.
hgt_grp = np.where(hgt > hgt_ref, "Tall", "Short")   # 기준 키 대비 Tall, Short.

df_dat['ht_g'] = hgt_grp            # data frame에 변수 추가된 것임.
df_dat['gndr'] = gndr               #   이것은 위에 1/0 자료를 추가. 성별임.

df_dat_tall = df_dat[df_dat['ht_g'] == 'Tall']  # 키 그룹별 자료 분리, 이게 데이터 셋.
df_dat_shrt = df_dat[df_dat['ht_g'] == 'Short'] # 키 그룹별 자료 분리

hgt_tall = hgt[hgt_grp=='Tall']  # 키 그룹별 분리, 배열은 hgt 하나임.
hgt_shrt = hgt[hgt_grp=='Short'] # 키 그룹별 분리

mf_tall = df_dat[df_dat['ht_g'] == 'Tall']['gndr']  # Tall 그룹 성별을 1/0으로 받음.
                                                    # 의미 없는 상황.
mf_shrt = df_dat[df_dat['ht_g'] == 'Short']['gndr']  # Tall 그룹 성별을 1/0으로 받음.

n_all = len(df_dat)
n_tall = len(df_dat_tall)
n_shrt = len(df_dat_shrt)

df_dat
#df_dat_tall
#df_dat_shrt

,i,gender,ht,gn,hgt,ht_g,gndr
0,1,1,159.9,1,159.9,Short,1
1,2,2,157.5,2,157.5,Short,0
2,3,2,158.0,2,158.0,Short,0
3,4,2,154.2,2,154.2,Short,0
4,5,1,163.3,1,163.3,Tall,1
...,...,...,...,...,...,...,...
20123,20124,2,157.3,2,157.3,Short,0
20124,20125,1,175.9,1,175.9,Tall,1
20125,20126,1,175.6,1,175.6,Tall,1
20126,20127,2,158.0,2,158.0,Short,0


### 남자(성별=1) 비율에 대한 추론.
전체에서 남자 비율, Tall 그룹에서 남자 비율, Short 그룹에서 남자 비율.

$ 100(1-\alpha ) $% 신뢰구간
$$ \hat p \pm z_{\alpha\over 2} \times SE  $$
$ SE = \sqrt{p (1-p) \over n } $

$ SE $ 추정 옵션. 1) $ \hat p $ 사용. 2) $ 1\over 2 $ 사용, 3) $ p_0 $ 사용

#### 정규분포 임계치

In [81]:
# 유의(신뢰)수준, 정규분포 임계치, 양방향, 우측값 (양수)
alpha = 0.05                          # significance level, two side.
zcv_r = stats.norm.ppf( 1- alpha/2 )  # critical value on the right, two side
zcv_r

np.float64(1.959963984540054)

#### 남 비율(전체, 옵션1, 옵션2)

In [82]:
# 3. (성별=1, Male) 비율, 비율의 표준오차, 모비율 신뢰구간 추정
# 3.1 대상 - 전체, Total
# 추정치: 비율, 비율의 표준오차 (옵션1, 옵션2)
phat_m = (gndr== 1).mean()                # 이렇게 해도 되는군... gndr = 1인 그룹 비율.
se_phat_m = np.sqrt(phat_m * (1 - phat_m) / n_all )  # 옵션 1.
se_max = np.sqrt( 0.5 * (1 - 0.5 ) / n_all )         # 옵션 2임. 최대 표준오차.

# 신뢰구간, right, left, (옵션1).
ci_r = phat_m + zcv_r * se_phat_m
ci_l = phat_m - zcv_r * se_phat_m

# 신뢰구간, right, left, (옵션2).
ci_ro2 = phat_m + zcv_r * se_max
ci_lo2 = phat_m - zcv_r * se_max

print("Male proportion: ")
print( "    phat,    se of phat,       confidence interval ")
print( " ALL  (옵션1):", phat_m, se_phat_m,   ci_l, ci_r )
# se max,se_max,
print( " ALL  (옵션2):", phat_m, se_max,     ci_lo2, ci_ro2 )

Male proportion: 
    phat,    se of phat,       confidence interval 
 ALL  (옵션1): 0.44415739268680443 0.003502225070661729 0.4372931576825542 0.45102162769105464
 ALL  (옵션2): 0.44415739268680443 0.003524274215216256 0.4372499421533374 0.45106484322027146


#### 남 비율(Tall, 옵션1)

In [83]:
# 3. (성별=1, Male) 비율, 비율의 표준오차, 모비율 신뢰구간 추정
# 3.2. 대상 - 큰 키 그룹, Tall
# 추정치(그룹별): 비율, 비율의 표준오차(옵션1), Tall - Short group, by height
phat_mt = mf_tall.mean()         # 이렇게 되는군...  위, Tall group의 성별임. mf.
se_phat_mt = np.sqrt(phat_mt * (1 - phat_mt) / n_tall )  # 옵션1.
ci_rt = phat_mt + zcv_r * se_phat_mt
ci_lt = phat_mt - zcv_r * se_phat_mt

print(" Tall (옵션1):", phat_mt, se_phat_mt, ci_lt, ci_rt )


 Tall (옵션1): 0.8089032527105922 0.004014397312994046 0.8010351785574895 0.8167713268636948


#### 남 비율(Short, 옵션1)

In [84]:
# 3. (성별=1, Male) 비율, 비율의 표준오차, 모비율 신뢰구간 추정
# 3.3. 대상 - 작은 키 그룹, Short
# 추정치(그룹별): 비율, 비율의 표준오차(옵션1),
phat_ms = mf_shrt.mean()         # 이렇게 재확인..  위, Short group의 성별임. mf.
se_phat_ms = np.sqrt(phat_ms * (1 - phat_ms) / n_shrt )  # 옵션 1임.
ci_rs = phat_ms + zcv_r * se_phat_ms
ci_ls = phat_ms - zcv_r * se_phat_ms

print("Short (옵션1):", phat_ms, se_phat_ms, ci_ls, ci_rs )
print(f"Short (옵션1):, {phat_ms:.6f}, {se_phat_ms:.6f}, {ci_ls:.6f}, {ci_rs:.6f}" )

Short (옵션1): 0.1120918754745634 0.003073499889155179 0.1060679263853314 0.1181158245637954
Short (옵션1):, 0.112092, 0.003073, 0.106068, 0.118116


### 가설검정, 검정통계량
가설(양측)
\begin{align}
 H_0 &: p = p_0 \quad \text{vs.} \quad
 H_A : p \ne p_0
\end{align}
검정통계량
\begin{align}
 T_0 & = { \hat p - p_0 \over SE_0}  \\
 & \approx N(0,1)
\end{align}

#### $ H_0 : p=p_0 \ vs. \ H_A: p \ne p_0 $

In [85]:
# 가설검정. 귀무가설
# 귀무가설 H0: p=p0, 대립가설 HA: p is not p0

p_zero = 0.45
print(" H0: p =", p_zero, " vs. HA: p is not ",  p_zero )

 H0: p = 0.45  vs. HA: p is not  0.45


#### 남 비율(전체, 옵션3)

In [86]:
# 전체
se_zero = np.sqrt( p_zero * (1 - p_zero) / n_all)   # 옵션 3.
t_0m = np.abs( ( phat_m - p_zero ) / se_zero )       # 옵션 3.

#t_0m = np.abs( ( phat_m - p_zero ) / se_phat_m )       # 옵션 1.

yn_h0m = " 'reject h0' " if t_0m > zcv_r else " 'fail to reject h0' "   # 이건 되는 군.
pvalm = 2*( 1 - stats.norm.cdf( t_0m ) )

print(" 검정통계치, 임계치, 검정결과, p값 ")
print(" 전체 (옵션3):", t_0m , zcv_r, yn_h0m, pvalm)


 검정통계치, 임계치, 검정결과, p값 
 전체 (옵션3): 1.6661703746316636 1.959963984540054  'fail to reject h0'  0.09567948481245514


#### 남 비율(Tall, 옵션3)

In [87]:
# Tall group
se_zero = np.sqrt( p_zero * (1 - p_zero) / n_tall)   # 옵션 3.
t_0mt = np.abs( ( phat_mt - p_zero ) / se_zero )       # 옵션 3.

#t_0mt = np.abs( ( phat_mt - p_zero ) / se_phat_mt )       # 옵션 1.

yn_h0 = " 'reject h0' " if t_0mt > zcv_r else " 'fail to reject h0' "   # 이건 되는 군.
pvalmt = 2*( 1 - stats.norm.cdf( t_0mt ) )

print(" Tall (옵션3): ", t_0mt , zcv_r, yn_h0, pvalmt)


 Tall (옵션3):  70.65524029352167 1.959963984540054  'reject h0'  0.0


#### 남 비율(Short, 옵션3)

In [88]:
# Short group
se_zero = np.sqrt( p_zero * (1 - p_zero) / n_shrt)   # 옵션 3.
t_0ms = np.abs( ( phat_ms - p_zero ) / se_zero )       # 옵션 3.

#t_0ms = np.abs( ( phat_ms - p_zero ) / se_phat_ms )       # 옵션 1.

yn_h0 = " 'reject h0' " if t_0mt > zcv_r else " 'fail to reject h0' "   # 이건 되는 군.
pvalms = 2*( 1 - stats.norm.cdf( t_0ms ) )

print("Short (옵션3):", t_0ms , zcv_r, yn_h0, pvalms)


Short (옵션3): 69.71864104262316 1.959963984540054  'reject h0'  0.0


### 모듈과 비교, 계산 결과 확인
모듈 디폴트, 옵션1 사용하는군.

In [89]:
# 계산 확인, 재확인, 모듈 반환 결과와 비교.
# 모듈 statsmodels.stats.proportion
# 처음에 모듈 import함.

count_mf = gndr.sum()          # number of male, male=1, female=0, overall,
count_mf_tall = mf_tall.sum()  # number of male among tall group, male=1, female=0
#    n_tall = len(mf_tall)    # 처음에 있음

count_mf_shrt = mf_shrt.sum()  # number of male among shrt group, male=1, female=0
#    n_shrt = len(mf_shrt)
ci_all = proportion_confint(count_mf, n_all, alpha=0.05, method="normal")
ci_tall = proportion_confint(count_mf_tall, n_tall, alpha=0.05, method="normal")
ci_shrt = proportion_confint(count_mf_shrt, n_shrt, alpha=0.05, method="normal")

print(f"전체 그룹 1의 비율: {count_mf / n_all:.4f} (95% CI: {ci_all})")
print(f"Tall 그룹 1의 비율: {count_mf_tall / n_tall:.4f} (95% CI: {ci_tall})")
print(f"Shrt 그룹 1의 비율: {count_mf_shrt / n_shrt:.4f} (95% CI: {ci_shrt})")
#    print(hgt_ref, len(dat) , len(hgt_tall), len(hgt_shrt))
#    print(" overall \n ", phat_m, se_phat_m,
#             " \n by group tall \n ", phat_mt, se_phat_mt
#             )

#    z0, pv = proportions_ztest(count_mf_tall, n_tall, p_zero, 'two-sided', p_zero)
z0a, pva = proportions_ztest(count_mf , n_all, p_zero, 'two-sided')
z0t, pvt = proportions_ztest(count_mf_tall, n_tall, p_zero, 'two-sided')
z0s, pvs = proportions_ztest(count_mf_shrt, n_shrt, p_zero, 'two-sided')
            # 옵션1
print("\n모듈 결과: statsmodels.stats.proportion ")
print(f" All, 그룹 1의 비율: {count_mf / n_all:.4f} (95% CI: {ci_all})")
print(" 검통 (옵1): ", z0a, " p값 ", pva)
print(f"Tall, 그룹 1의 비율: {count_mf_tall / n_tall:.4f} (95% CI: {ci_tall})")
print(" 검통 (옵1): ", z0t, " p값 ", pvt)
print(f"Short, 그룹 1의 비율: {count_mf_shrt / n_shrt:.4f} (95% CI: {ci_shrt})")
print(" 검통 (옵1): ", z0s, " p값 ", pvs)




전체 그룹 1의 비율: 0.4442 (95% CI: (0.43729315768255417, 0.4510216276910547))
Tall 그룹 1의 비율: 0.8089 (95% CI: (0.8010351785574895, 0.8167713268636948))
Shrt 그룹 1의 비율: 0.1121 (95% CI: (0.1060679263853314, 0.1181158245637954))

모듈 결과: statsmodels.stats.proportion 
 All, 그룹 1의 비율: 0.4442 (95% CI: (0.43729315768255417, 0.4510216276910547))
 검통 (옵1):  -1.6682558074691776  p값  0.0952649566427439
Tall, 그룹 1의 비율: 0.8089 (95% CI: (0.8010351785574895, 0.8167713268636948))
 검통 (옵1):  89.4040187673682  p값  0.0
Short, 그룹 1의 비율: 0.1121 (95% CI: (0.1060679263853314, 0.1181158245637954))
 검통 (옵1):  -109.94245541304323  p값  0.0


## 두 그룹 (Tall, Short)의 남자(성별=1) 비율은 같은가?
즉, 키가 크고 작고와 무관하게, 남자의 비율은 대략 비슷한가?

별도 주제, 평균차이 추론 섹션

In [90]:
print("Computing OK")


Computing OK
